In [1]:
from transformers import AutoModel, AutoTokenizer, AutoConfig
import torch
import torch.nn as nn

class RewardModel(nn.Module):
    def __init__(self, model_name="TinyLlama/TinyLlama-1.1B-Chat"):
        super().__init__()
        self.base_model = AutoModel.from_pretrained(model_name)
        self.hidden_size = self.base_model.config.hidden_size
        self.reward_head = nn.Linear(self.hidden_size, 1)

    def forward(self, input_ids, attention_mask):
        outputs = self.base_model(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state  # shape: [B, T, H]
        cls_token_embedding = last_hidden_state[:, 0, :]  # Take first token's embedding
        reward = self.reward_head(cls_token_embedding)
        return reward.squeeze(-1)  # shape: [B]


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # <-- define device here

model = model.to(device)


In [4]:
from torch.utils.data import Dataset, DataLoader

class MyRankingDataset(Dataset):
    def __init__(self, data, tokenizer, max_length=512):
        """
        data: list of dicts with keys "chosen_text" and "rejected_text"
        tokenizer: Hugging Face tokenizer
        """
        self.data = data
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        chosen_enc = self.tokenizer(item["chosen_text"], max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")
        rejected_enc = self.tokenizer(item["rejected_text"], max_length=self.max_length, truncation=True, padding="max_length", return_tensors="pt")

        return {
            "input_ids_chosen": chosen_enc["input_ids"].squeeze(0),           # shape: [max_length]
            "attention_mask_chosen": chosen_enc["attention_mask"].squeeze(0), # shape: [max_length]
            "input_ids_rejected": rejected_enc["input_ids"].squeeze(0),
            "attention_mask_rejected": rejected_enc["attention_mask"].squeeze(0)
        }

# Example data (replace with your actual data)
data = [
    {"chosen_text": "I love sunny days.", "rejected_text": "I hate sunny days."},
    {"chosen_text": "The movie was great!", "rejected_text": "The movie was boring."},
    # Add more examples...
]

dataset = MyRankingDataset(data, tokenizer, max_length=64)
dataloader = DataLoader(dataset, batch_size=2, shuffle=True)


In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

# Load model and tokenizer
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.train()

# Define ranking loss (hinge loss)
def ranking_loss(chosen, rejected, margin=0.0):
    return torch.nn.functional.relu(margin - (chosen - rejected)).mean()

# Reward extraction helper
def get_reward(output, attention_mask):
    # output.hidden_states[-1]: [batch, seq_len, hidden_dim]
    last_hidden = output.hidden_states[-1]
    last_token_idx = attention_mask.sum(dim=1) - 1  # [batch]
    reward = last_hidden[torch.arange(last_hidden.size(0)), last_token_idx]  # [batch, hidden_dim]
    return reward.mean(dim=-1)  # scalar reward per sample

# Hyperparameters
num_epochs = 3
learning_rate = 2e-5

# Optimizer
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

# Example dataloader placeholder — replace with your actual dataloader!
# dataloader = ...

for epoch in range(num_epochs):
    total_loss = 0.0

    for batch in tqdm(dataloader, desc=f"Epoch {epoch+1}"):
        input_ids_chosen = batch["input_ids_chosen"].to(device)
        attention_mask_chosen = batch["attention_mask_chosen"].to(device)
        input_ids_rejected = batch["input_ids_rejected"].to(device)
        attention_mask_rejected = batch["attention_mask_rejected"].to(device)

        # Forward pass with gradients (remove torch.no_grad)
        output_chosen = model(input_ids=input_ids_chosen, attention_mask=attention_mask_chosen, output_hidden_states=True)
        output_rejected = model(input_ids=input_ids_rejected, attention_mask=attention_mask_rejected, output_hidden_states=True)

        chosen_reward = get_reward(output_chosen, attention_mask_chosen)
        rejected_reward = get_reward(output_rejected, attention_mask_rejected)

        loss = ranking_loss(chosen_reward, rejected_reward)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(dataloader)
    print(f"Epoch {epoch+1} - Average Loss: {avg_loss:.4f}")


Epoch 1: 100%|██████████| 1/1 [00:37<00:00, 37.82s/it]


Epoch 1 - Average Loss: 0.0000


Epoch 2: 100%|██████████| 1/1 [00:27<00:00, 27.51s/it]


Epoch 2 - Average Loss: 0.0000


Epoch 3: 100%|██████████| 1/1 [00:24<00:00, 24.59s/it]

Epoch 3 - Average Loss: 0.0000


In [6]:
!pip install trl


In [1]:
from transformers import AutoTokenizer
from trl import AutoModelForCausalLMWithValueHead

tokenizer = AutoTokenizer.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")
model = AutoModelForCausalLMWithValueHead.from_pretrained("TinyLlama/TinyLlama-1.1B-Chat-v1.0")


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
!pip install -U trl


In [3]:
import trl
print(trl.__version__)


0.19.1


In [ ]:
import torch
from torch import nn
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from trl import PPOTrainer, PPOConfig

# === 1. Load model and tokenizer ===
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id)
ref_model = AutoModelForCausalLM.from_pretrained(model_id)
ref_model.eval()
for param in ref_model.parameters():
    param.requires_grad = False

# === 2. Dummy reward and value models ===
class DummyRewardModel(nn.Module):
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        # Returns reward tensor of shape (batch_size,)
        # Some PPO calls can use other args, so take **kwargs
        batch_size = input_ids.shape[0]
        return torch.ones(batch_size, device=input_ids.device)

class DummyValueHead(nn.Module):
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        # PPOTrainer expects (values,) output, shape (batch_size, seq_len)
        batch_size, seq_len = input_ids.shape
        return torch.zeros((batch_size, seq_len), device=input_ids.device),

reward_model = DummyRewardModel()
value_model = DummyValueHead()

# === 3. Dataset (dummy list of prompts) ===
train_dataset = [
    "Hello, how are you?",
    "What is the weather today?",
    "Tell me a joke.",
    "Explain quantum physics."
]

# === 4. PPO Config ===
ppo_config = PPOConfig(
    batch_size=2,
    learning_rate=1e-5,
    mini_batch_size=2,
    optimize_cuda_cache=False,
    log_with=None,
    project_kwargs={},
    accelerator_kwargs={},
    remove_unused_columns=False,  # Important for custom columns
)

# === 5. PPOTrainer expects: model, ref_model, reward_model, tokenizer, dataset, config. ===
ppo_trainer = PPOTrainer(
    model=model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=train_dataset,
    config=ppo_config,
    reward_model=reward_model,
    value_model=value_model,
)

# === 6. Training loop ===
num_epochs = 3
for epoch in range(num_epochs):
    print(f"Epoch {epoch + 1}")
    for prompt in train_dataset:
        # Tokenize prompt (as batch)
        batch = tokenizer([prompt], return_tensors="pt", padding=True, truncation=True)
        batch = {k: v.to(model.device) for k, v in batch.items()}

        # Generate a response using the model
        with torch.no_grad():
            response_ids = ppo_trainer.generate(batch["input_ids"])
        
        # Generate reward (dummy)
        rewards = reward_model(response_ids)
        
        # Call PPO step using dummy reward
        ppo_trainer.step([prompt], response_ids, rewards)


C:\Users\gadep\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
